### 1. Inductive vs. Deductive Reasoning

* **Inductive Reasoning:** Making broad generalizations from specific observations.
  * *Example:* Observing that the local coffee shop was busy the last three Saturdays at 10:00 AM, and concluding that the coffee shop is always busy on Saturday mornings.
* **Deductive Reasoning:** Starting with a general rule or premise and moving to a specific, guaranteed conclusion.
  * *Example:* Starting with the rule that all mammals have backbones, noting that a whale is a mammal, and concluding that a whale must have a backbone.

### 2. Dataset Pre-processing

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

df_app = pd.read_csv('Credit_Card.csv')
df_lbl = pd.read_csv('Credit_card_label.csv')

df = pd.merge(df_app, df_lbl, left_on='Ind_ID', right_on='ID')

df['Age'] = df['Birthday_count'] / -365.25
df['Years_Employed'] = df['Employed_days'] / -365.25
df['Years_Employed'] = df['Years_Employed'].apply(lambda x: 0 if x < 0 else x)

df['Type_Occupation'] = df['Type_Occupation'].fillna('Unknown')
df['Annual_income'] = df['Annual_income'].fillna(df['Annual_income'].median())

df.drop(['Ind_ID', 'ID', 'Birthday_count', 'Employed_days', 'Mobile_phone'], axis=1, inplace=True)

categorical_cols = ['Gender', 'Car_owner', 'Propert_owner', 'Type_Income', 'Education', 'Marital_status', 'Housing_type', 'Type_Occupation']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df.drop('Label', axis=1)
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

### 3. Decision Tree Model

**How it was tuned:** I restricted the maximum depth to 7 to limit tree complexity. Setting a cap prevents the tree from creating infinite branches that simply memorize edge-case training samples instead of looking for generalized patterns.

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=7, random_state=42)
dt_model.fit(X_train, y_train)

dt_preds = dt_model.predict(X_test)
print(classification_report(y_test, dt_preds))
ConfusionMatrixDisplay.from_predictions(y_test, dt_preds)

### 4. Random Forest Model

**How it was tuned:** I set the number of estimators to 100 to give the model a wide pool of trees to average calculations over. I restricted the depth of individual trees to 7 to match our decision tree constraints.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
print(classification_report(y_test, rf_preds))
ConfusionMatrixDisplay.from_predictions(y_test, rf_preds)

### 5. XGBoost Model

**How it was tuned:** I slowed the learning rate down to 0.05 so that the sequential gradient boosting trees correct mistakes slowly rather than swinging wildly. I held the tree depth to 4 to ensure high execution speeds and prevent overfitting.

In [ ]:
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)
print(classification_report(y_test, xgb_preds))
ConfusionMatrixDisplay.from_predictions(y_test, xgb_preds)